# Pipeline dữ liệu thời tiết và tín hiệu lũ Điện Biên

Notebook này chỉ điều phối pipeline đã được kiểm thử. Dữ liệu nghiệp vụ nằm trong Parquet/manifest; notebook không hard-code tọa độ hay logic cảnh báo.

In [40]:
%pip install -q pandas pyarrow requests numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [41]:
import subprocess
import sys
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "data").is_dir() else cwd.parent
data_dir = repo_root / "data"
locations_path = data_dir / "dien_bien_locations.parquet"
if not locations_path.exists():
    raise FileNotFoundError(f"Không tìm thấy {locations_path}")

def run_data_script(script_name, *arguments):
    command = [sys.executable, str(data_dir / script_name), *map(str, arguments)]
    subprocess.run(command, cwd=repo_root, check=True)

# P0–P2: elevation → forecast snapshot → cảnh báo thời tiết MVP.
for script_name in [
    "download_elevation.py",
    "download_forecast.py",
    "alert_rules.py",
    "verify_weather_alert_mvp.py",
]:
    run_data_script(script_name)

# P3: node sông OSM → GloFAS daily → trend signal bổ trợ.
river_points_path = data_dir / "river_points.parquet"
if not river_points_path.exists():
    run_data_script("build_river_points.py")
for script_name in [
    "download_flood.py",
    "flood_signal.py",
    "verify_flood_signal.py",
]:
    run_data_script(script_name)

In [42]:
overview_files = sorted(
    (data_dir / "alerts").glob(
        "snapshot_date=*/snapshot_time=*/new_admin_risk_overview.parquet"
    )
)
if not overview_files:
    raise FileNotFoundError("Chưa có new_admin_risk_overview.parquet")

risk_overview = pd.read_parquet(overview_files[-1])
flood_signals = pd.read_parquet(data_dir / "flood_signals.parquet")

display(
    risk_overview.sort_values(
        ["severity_rank", "new_admin_unit"],
        ascending=[False, True],
    )
)
display(
    flood_signals.loc[flood_signals["is_representative_grid_cell"]]
    .sort_values("peak_change_percent", ascending=False)
)

,new_admin_unit,province,coverage_status,severity,severity_rank,hazard_type,confidence,valid_from,valid_to,alert_count
43,Phường Điện Biên Phủ,Điện Biên,covered,warning,1,prolonged_rain_signal,medium,2026-07-18 00:00:00+07:00,2026-07-20 23:00:00+07:00,1
0,Xã Mường Nhé,Điện Biên,covered,warning,1,heavy_rain,high,2026-07-19 10:00:00+07:00,2026-07-20 09:00:00+07:00,3
29,Xã Mường Phăng,Điện Biên,covered,warning,1,prolonged_rain_signal,medium,2026-07-18 00:00:00+07:00,2026-07-20 23:00:00+07:00,1
2,Xã Mường Toong,Điện Biên,covered,warning,1,heavy_rain,high,2026-07-19 10:00:00+07:00,2026-07-20 09:00:00+07:00,3
11,Xã Mường Tùng,Điện Biên,covered,warning,1,heavy_rain,high,2026-07-18 15:00:00+07:00,2026-07-19 14:00:00+07:00,1
5,Xã Nà Hỳ,Điện Biên,covered,warning,1,prolonged_rain_signal,medium,2026-07-18 00:00:00+07:00,2026-07-20 23:00:00+07:00,1
3,Xã Nậm Kè,Điện Biên,covered,warning,1,heavy_rain,high,2026-07-19 10:00:00+07:00,2026-07-20 09:00:00+07:00,2
12,Xã Pa Ham,Điện Biên,covered,warning,1,prolonged_rain_signal,medium,2026-07-18 00:00:00+07:00,2026-07-20 23:00:00+07:00,1
4,Xã Quảng Lâm,Điện Biên,covered,warning,1,prolonged_rain_signal,medium,2026-07-18 00:00:00+07:00,2026-07-20 23:00:00+07:00,1
33,Xã Sam Mứn,Điện Biên,covered,warning,1,prolonged_rain_signal,medium,2026-07-18 00:00:00+07:00,2026-07-20 23:00:00+07:00,1


,river_point_id,river_name,point_name,snapshot_at,baseline_discharge_m3s,peak_discharge_m3s,peak_valid_date,peak_change_percent,horizon_end_discharge_m3s,horizon_change_percent,...,confidence,is_official_warning,message_vi,model,signal_version,grid_latitude,grid_longitude,grid_cell_id,grid_point_count,is_representative_grid_cell
6,osm-way-470292947-node-6499735491,Nậm Lay,Nậm Lay 02,2026-07-17 20:00:00+07:00,3.62,8.46,2026-07-24,133.701657,3.14,-13.259669,...,high,False,Tín hiệu mô phỏng GloFAS để theo dõi xu hướng;...,glofas_v4_seamless,glofas-trend-1.0,22.025002,103.175020,"22.02500,103.17502",2,True
0,osm-way-279788696-node-2293741467,Nậm Rốm,Nậm Rốm 02,2026-07-17 20:00:00+07:00,2.06,4.81,2026-07-25,133.495146,2.66,29.126214,...,high,False,Tín hiệu mô phỏng GloFAS để theo dõi xu hướng;...,glofas_v4_seamless,glofas-trend-1.0,21.375000,103.025024,"21.37500,103.02502",2,True
8,osm-way-67504081-node-4202449088,Nậm Mức,Nậm Mức 01,2026-07-17 20:00:00+07:00,3.29,7.28,2026-07-25,121.276596,4.09,24.316109,...,high,False,Tín hiệu mô phỏng GloFAS để theo dõi xu hướng;...,glofas_v4_seamless,glofas-trend-1.0,21.975006,103.275024,"21.97501,103.27502",2,True
10,osm-way-67504081-node-4202449309,Nậm Mức,Nậm Mức 03,2026-07-17 20:00:00+07:00,3.70,7.95,2026-07-24,114.864865,3.17,-14.324324,...,high,False,Tín hiệu mô phỏng GloFAS để theo dõi xu hướng;...,glofas_v4_seamless,glofas-trend-1.0,22.025002,103.275024,"22.02500,103.27502",2,True
1,osm-way-279788696-node-2293741794,Nậm Rốm,Nậm Rốm 03,2026-07-17 20:00:00+07:00,1.89,3.92,2026-07-25,107.407407,2.55,34.920635,...,high,False,Tín hiệu mô phỏng GloFAS để theo dõi xu hướng;...,glofas_v4_seamless,glofas-trend-1.0,21.325005,103.025024,"21.32501,103.02502",2,True
4,osm-way-470292947-node-4644971211,Nậm Lay,Nậm Lay 04,2026-07-17 20:00:00+07:00,2759.58,5456.58,2026-07-21,97.732264,3053.92,10.666116,...,high,False,Tín hiệu mô phỏng GloFAS để theo dõi xu hướng;...,glofas_v4_seamless,glofas-trend-1.0,22.075005,103.175020,"22.07501,103.17502",1,True
5,osm-way-470292947-node-6499734268,Nậm Lay,Nậm Lay 01,2026-07-17 20:00:00+07:00,54.72,105.66,2026-07-24,93.092105,47.19,-13.760965,...,high,False,Tín hiệu mô phỏng GloFAS để theo dõi xu hướng;...,glofas_v4_seamless,glofas-trend-1.0,21.975006,103.125000,"21.97501,103.12500",1,True
